# Lab 1: Vectorless RAG

See `lab1_vectorless_rag.md` for full writeup, architecture diagrams, and concepts.

## Setup

In [ ]:
!pip install -q pageindex openai python-dotenv pymupdf

In [ ]:
import os
import json
import base64
import re
import fitz  # PyMuPDF — renders PDF pages as images
from openai import OpenAI
from dotenv import load_dotenv

# Load API keys from the .env file in the project root
load_dotenv("../.env")

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

print("Keys loaded.")

In [ ]:
def call_vlm(prompt, image_paths=None, model="meta-llama/llama-4-scout-17b-16e-instruct"):
    """Call a vision model via OpenRouter.

    Args:
        prompt: The text prompt to send to the model.
        image_paths: Optional list of image file paths to include as visual context.
        model: The VLM model identifier on OpenRouter.

    Returns:
        The model's text response.
    """
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
    )
    if image_paths:
        # Build a multimodal message: text prompt + base64-encoded images
        content = [{"type": "text", "text": prompt}]
        for img in image_paths:
            if os.path.exists(img):
                # Read image bytes and encode as base64 data URL
                with open(img, "rb") as f:
                    b64 = base64.b64encode(f.read()).decode()
                content.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}})
        msgs = [{"role": "user", "content": content}]
    else:
        # Text-only prompt (no images)
        msgs = [{"role": "user", "content": prompt}]
    resp = client.chat.completions.create(model=model, messages=msgs, temperature=0, max_tokens=1024)
    return resp.choices[0].message.content.strip()

---
## Step 1 — Render PDF Pages

In [ ]:
PDF_PATH = "data/synthetic_medicare_plus_policy_detailed.pdf"

def extract_page_images(pdf_path, out="pdf_images"):
    """Render each PDF page as a JPEG image.

    The VLM will later read these images to answer questions,
    so we render at 2x resolution for better clarity.

    Returns:
        dict mapping 1-based page number to image file path
    """
    os.makedirs(out, exist_ok=True)
    doc = fitz.open(pdf_path)
    imgs = {}
    for i in range(len(doc)):
        # Render page at 2x zoom for higher quality
        pix = doc.load_page(i).get_pixmap(matrix=fitz.Matrix(2.0, 2.0))
        path = os.path.join(out, f"page_{i+1}.jpg")
        open(path, "wb").write(pix.tobytes("jpeg"))
        imgs[i+1] = path
    doc.close()
    return imgs

page_images = extract_page_images(PDF_PATH)
print(f"Rendered {len(page_images)} pages.")

---
## Step 2 — Build Document Tree (with caching)

The tree is saved to `cache/` as a JSON file keyed by the PDF filename. On repeat runs with the same PDF, the cached tree is loaded instantly — no need to re-process with PageIndex.

In [ ]:
from pageindex import PageIndexClient
from pageindex import utils
import time

# --- Caching helpers ---
# PageIndex processing takes time. We cache the tree as JSON
# so repeat runs with the same PDF skip the API call entirely.

CACHE_DIR = "cache"
os.makedirs(CACHE_DIR, exist_ok=True)

def get_cache_path(pdf_path):
    """Generate a cache file path based on the PDF filename.

    e.g. 'data/policy.pdf' -> 'cache/policy_tree.json'
    """
    pdf_name = os.path.splitext(os.path.basename(pdf_path))[0]
    return os.path.join(CACHE_DIR, f"{pdf_name}_tree.json")

def load_cached_tree(pdf_path):
    """Load tree from cache if it exists, otherwise return None."""
    cache_path = get_cache_path(pdf_path)
    if os.path.exists(cache_path):
        with open(cache_path, "r") as f:
            tree = json.load(f)
        print(f"Loaded tree from cache: {cache_path}")
        return tree
    return None

def save_tree_to_cache(pdf_path, tree):
    """Save the PageIndex tree to a JSON file for future reuse."""
    cache_path = get_cache_path(pdf_path)
    with open(cache_path, "w") as f:
        json.dump(tree, f, indent=2)
    print(f"Tree saved to cache: {cache_path}")

In [ ]:
pi = PageIndexClient(api_key=PAGEINDEX_API_KEY)

# Try loading from cache first — avoids re-processing the same PDF
tree = load_cached_tree(PDF_PATH)

if tree is None:
    # Cache miss — submit PDF to PageIndex for tree generation
    print("Cache miss — submitting to PageIndex...")
    result = pi.submit_document(PDF_PATH)
    doc_id = result["doc_id"]
    print(f"Submitted: {doc_id}")

    # Poll until PageIndex finishes processing (max 5 min)
    # PageIndex parses the PDF asynchronously — we check every 5 seconds
    print("Waiting for PageIndex to process...")
    elapsed = 0
    while elapsed < 300:
        if pi.is_retrieval_ready(doc_id):
            break
        time.sleep(5)
        elapsed += 5
        print(f"  {elapsed}s...")
    else:
        raise TimeoutError(f"PageIndex did not finish within {elapsed}s. Try again later.")

    print(f"Ready after {elapsed}s")
    # Fetch the full hierarchical tree with per-node summaries
    tree = pi.get_tree(doc_id, node_summary=True)["result"]
    save_tree_to_cache(PDF_PATH, tree)

# Display the tree structure (titles + summaries, no full text)
utils.print_tree(tree, exclude_fields=["text"])

---
## Step 3 — Ask a Question

In [ ]:
QUERY = "What is the GST rate applied to the premium?"

---
## Step 4 — VLM Finds Relevant Sections

In [ ]:
# Remove full text from tree — the VLM only needs titles + summaries
# to decide which sections are relevant. This keeps the prompt small.
tree_slim = utils.remove_fields(tree.copy(), fields=["text"])

# Ask the VLM to reason over the tree and pick relevant nodes
search_prompt = f"""
You are given a question and a document tree.
Each node has: node_id, title, summary.
Find all nodes likely to contain the answer.

Question: {QUERY}

Document tree:
{json.dumps(tree_slim, indent=2)}

Reply in this JSON format ONLY:
{{
    "thinking": "<your reasoning>",
    "node_list": ["node_id_1", "node_id_2"]
}}
"""

raw_result = call_vlm(search_prompt)
print(raw_result)

In [ ]:
def parse_json(text):
    """Extract JSON from VLM response, handling markdown code fences."""
    # Strip markdown ```json ... ``` wrappers if present
    text = re.sub(r"```json\s*|\s*```", "", text.strip())
    # Find the first { and last } to extract just the JSON object
    s, e = text.find("{"), text.rfind("}")
    if s != -1 and e != -1:
        text = text[s:e+1]
    return json.loads(text)

result = parse_json(raw_result)

# Build a mapping from node_id -> node info (title, page range, etc.)
# This lets us look up which pages each retrieved node spans
node_map = utils.create_node_mapping(tree, include_page_ranges=True, max_page=len(page_images))

print("Reasoning:", result["thinking"], "\n")
print("Retrieved nodes:")
for nid in result["node_list"]:
    info = node_map[nid]
    # Format page range: single page "5" or range "5-7"
    pages = info['start_index'] if info['start_index'] == info['end_index'] else f"{info['start_index']}-{info['end_index']}"
    print(f"  {nid} | Pages {pages} | {info['node']['title']}")

---
## Step 5 — VLM Answers from Page Images

In [ ]:
# Map retrieved node IDs to their corresponding PDF page images.
# Each node spans one or more pages — we collect all unique page images.
def images_for_nodes(nodes, node_map, page_images):
    paths, seen = [], set()
    for nid in nodes:
        info = node_map[nid]
        # Iterate over all pages this node covers
        for p in range(info["start_index"], info["end_index"] + 1):
            # Skip duplicates (multiple nodes may reference the same page)
            if p not in seen and p in page_images:
                paths.append(page_images[p])
                seen.add(p)
    return paths

images = images_for_nodes(result["node_list"], node_map, page_images)
print(f"Using {len(images)} page image(s).")

In [ ]:
# Send the page images + question to the VLM for visual QA.
# The VLM reads the actual PDF rendering — preserving tables, figures, and layout.
answer_prompt = f"""
Answer the question based on the provided page images.

Question: {QUERY}

Rules:
- Answer only from the images
- If the answer isn't there, say so
- Be concise
"""

answer = call_vlm(answer_prompt, image_paths=images)
print(answer)

---
## Try It Yourself

Change `QUERY` and re-run from **Step 4**.